# GeoRules Interactive Explorer

Interactively create and visualize 3D geological models. Select a model type, adjust parameters with sliders, and generate models on demand.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import georules as gr

%matplotlib inline

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


## Model Builder

1. Choose a geology type from the dropdown
2. Adjust the parameters using sliders — the plot updates automatically

In [ ]:
# ── Shared grid widgets ──────────────────────────────────────────────
style = {'description_width': '140px'}
layout = widgets.Layout(width='420px')

w_nx = widgets.IntSlider(value=80, min=20, max=200, step=10, description='nx (grid X)', style=style, layout=layout)
w_ny = widgets.IntSlider(value=80, min=20, max=200, step=10, description='ny (grid Y)', style=style, layout=layout)
w_nz = widgets.IntSlider(value=30, min=5, max=80, step=5, description='nz (grid Z)', style=style, layout=layout)
w_xlen = widgets.IntSlider(value=2000, min=200, max=5000, step=200, description='x_len (m)', style=style, layout=layout)
w_ylen = widgets.IntSlider(value=2000, min=200, max=5000, step=200, description='y_len (m)', style=style, layout=layout)
w_zlen = widgets.IntSlider(value=80, min=10, max=300, step=10, description='z_len (m)', style=style, layout=layout)
w_depth = widgets.IntSlider(value=5000, min=1000, max=8000, step=100, description='top_depth (m)', style=style, layout=layout)
w_dip = widgets.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.5, description='dip (deg)', style=style, layout=layout)

grid_box = widgets.VBox([
    widgets.HTML('<b>Grid Parameters</b>'),
    widgets.HBox([widgets.VBox([w_nx, w_ny, w_nz, w_depth]),
                  widgets.VBox([w_xlen, w_ylen, w_zlen, w_dip])])
])

# ── Lobe widgets ─────────────────────────────────────────────────────
l_poro = widgets.FloatSlider(value=0.20, min=0.05, max=0.40, step=0.01, description='poro_ave', style=style, layout=layout)
l_perm = widgets.FloatSlider(value=1.5, min=0.1, max=5.0, step=0.1, description='perm_ave', style=style, layout=layout)
l_poro_std = widgets.FloatSlider(value=0.03, min=0.005, max=0.10, step=0.005, description='poro_std', style=style, layout=layout)
l_perm_std = widgets.FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05, description='perm_std', style=style, layout=layout)
l_ntg = widgets.FloatSlider(value=0.7, min=0.1, max=1.0, step=0.05, description='NTG', style=style, layout=layout)
l_rmin = widgets.IntSlider(value=15, min=5, max=50, step=1, description='rmin (lobe)', style=style, layout=layout)
l_rmax = widgets.IntSlider(value=25, min=10, max=60, step=1, description='rmax (lobe)', style=style, layout=layout)
l_asp = widgets.FloatSlider(value=1.5, min=1.0, max=5.0, step=0.1, description='aspect ratio', style=style, layout=layout)
l_m = widgets.IntSlider(value=100, min=10, max=500, step=10, description='n_lobes (m)', style=style, layout=layout)
l_upthin = widgets.Checkbox(value=False, description='upthinning', style=style, layout=layout)
l_bouma = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description='bouma_factor', style=style, layout=layout)

lobe_box = widgets.VBox([
    widgets.HTML('<b>Lobe Parameters</b>'),
    widgets.HBox([widgets.VBox([l_poro, l_perm, l_poro_std, l_perm_std, l_ntg, l_bouma]),
                  widgets.VBox([l_rmin, l_rmax, l_asp, l_m, l_upthin])])
])

# ── Gaussian widgets ─────────────────────────────────────────────────
g_poro = widgets.FloatSlider(value=0.15, min=0.05, max=0.40, step=0.01, description='poro_ave', style=style, layout=layout)
g_perm = widgets.FloatSlider(value=1.2, min=0.1, max=5.0, step=0.1, description='perm_ave', style=style, layout=layout)
g_poro_std = widgets.FloatSlider(value=0.03, min=0.005, max=0.10, step=0.005, description='poro_std', style=style, layout=layout)
g_perm_std = widgets.FloatSlider(value=0.4, min=0.05, max=2.0, step=0.05, description='perm_std', style=style, layout=layout)
g_ntg = widgets.FloatSlider(value=0.6, min=0.1, max=1.0, step=0.05, description='NTG', style=style, layout=layout)
g_fx = widgets.FloatSlider(value=2.5, min=0.5, max=10.0, step=0.5, description='facies_filt X', style=style, layout=layout)
g_fy = widgets.FloatSlider(value=5.0, min=0.5, max=15.0, step=0.5, description='facies_filt Y', style=style, layout=layout)
g_fz = widgets.FloatSlider(value=2.5, min=0.5, max=10.0, step=0.5, description='facies_filt Z', style=style, layout=layout)
g_sx = widgets.FloatSlider(value=1.5, min=0.5, max=5.0, step=0.5, description='sand_filt X', style=style, layout=layout)
g_sy = widgets.FloatSlider(value=2.5, min=0.5, max=8.0, step=0.5, description='sand_filt Y', style=style, layout=layout)
g_sz = widgets.FloatSlider(value=1.5, min=0.5, max=5.0, step=0.5, description='sand_filt Z', style=style, layout=layout)
g_nugget = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description='nugget', style=style, layout=layout)

gauss_box = widgets.VBox([
    widgets.HTML('<b>Gaussian Parameters</b>'),
    widgets.HBox([widgets.VBox([g_poro, g_perm, g_poro_std, g_perm_std, g_ntg, g_nugget]),
                  widgets.VBox([g_fx, g_fy, g_fz, g_sx, g_sy, g_sz])])
])

# ── Meandering Channel widgets ───────────────────────────────────────
c_width = widgets.IntSlider(value=80, min=20, max=300, step=10, description='channel_width (m)', style=style, layout=layout)
c_nch = widgets.IntSlider(value=5, min=1, max=20, step=1, description='n_channels', style=style, layout=layout)
c_dwr = widgets.FloatSlider(value=0.4, min=0.1, max=1.0, step=0.05, description='depth/width ratio', style=style, layout=layout)
c_meander = widgets.FloatSlider(value=0.8, min=0.0, max=3.0, step=0.1, description='meander_scale', style=style, layout=layout)
c_avulsion = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description='avulsion_prob', style=style, layout=layout)
c_poro = widgets.FloatSlider(value=0.3, min=0.05, max=0.40, step=0.01, description='poro_ave', style=style, layout=layout)

channel_box = widgets.VBox([
    widgets.HTML('<b>Meandering Channel Parameters</b>'),
    widgets.HBox([widgets.VBox([c_width, c_nch, c_dwr]),
                  widgets.VBox([c_meander, c_avulsion, c_poro])])
])

# ── Braided Channel widgets ──────────────────────────────────────────
b_bpw = widgets.IntSlider(value=500, min=100, max=1500, step=50, description='braidplain_width', style=style, layout=layout)
b_nch = widgets.IntSlider(value=6, min=1, max=20, step=1, description='n_channels', style=style, layout=layout)
b_nth = widgets.IntSlider(value=3, min=2, max=8, step=1, description='n_threads', style=style, layout=layout)
b_dwr = widgets.FloatSlider(value=0.15, min=0.05, max=0.5, step=0.05, description='depth/width ratio', style=style, layout=layout)
b_bar_poro = widgets.FloatSlider(value=0.7, min=0.1, max=1.0, step=0.05, description='bar_poro_factor', style=style, layout=layout)
b_poro = widgets.FloatSlider(value=0.3, min=0.05, max=0.40, step=0.01, description='poro_ave', style=style, layout=layout)

braided_box = widgets.VBox([
    widgets.HTML('<b>Braided Channel Parameters</b>'),
    widgets.HBox([widgets.VBox([b_bpw, b_nch, b_nth]),
                  widgets.VBox([b_dwr, b_bar_poro, b_poro])])
])

# ── View options ─────────────────────────────────────────────────────
w_view = widgets.Dropdown(
    options=['Cube Slices', 'Z Slices', 'X Slices', 'Y Slices'],
    value='Cube Slices', description='View', style=style, layout=layout
)
w_prop = widgets.Dropdown(
    options=['Porosity', 'Permeability', 'Facies'],
    value='Porosity', description='Property', style=style, layout=layout
)
view_box = widgets.VBox([
    widgets.HTML('<b>Visualization</b>'),
    widgets.HBox([w_view, w_prop])
])

print('Widgets ready.')

Widgets ready.


In [3]:
# ── Model selector & dynamic panel ───────────────────────────────────
model_selector = widgets.Dropdown(
    options=['Lobe', 'Gaussian', 'Meandering Channel', 'Braided Channel'],
    value='Lobe', description='Model Type', style=style, layout=layout
)

param_area = widgets.Output()
output_area = widgets.Output()

# Map model names to their parameter boxes and their widgets
param_panels = {
    'Lobe': lobe_box,
    'Gaussian': gauss_box,
    'Meandering Channel': channel_box,
    'Braided Channel': braided_box,
}

def update_params(*args):
    with param_area:
        clear_output(wait=True)
        display(param_panels[model_selector.value])

model_selector.observe(update_params, names='value')

# ── Auto-generate on any widget change ───────────────────────────────
def generate_model(*args):
    with output_area:
        clear_output(wait=True)
        plt.close('all')
        model_type = model_selector.value
        print(f'Generating {model_type} model...')

        try:
            if model_type == 'Lobe':
                layer = gr.LobeLayer(
                    nx=w_nx.value, ny=w_ny.value, nz=w_nz.value,
                    x_len=w_xlen.value, y_len=w_ylen.value, z_len=w_zlen.value,
                    top_depth=w_depth.value, dip=w_dip.value
                )
                layer.create_geology(
                    poro_ave=l_poro.value, perm_ave=l_perm.value,
                    poro_std=l_poro_std.value, perm_std=l_perm_std.value,
                    ntg=l_ntg.value,
                    rmin=l_rmin.value, rmax=l_rmax.value,
                    asp=l_asp.value, m=l_m.value,
                    upthinning=l_upthin.value,
                    bouma_factor=l_bouma.value
                )

            elif model_type == 'Gaussian':
                layer = gr.GaussianLayer(
                    nx=w_nx.value, ny=w_ny.value, nz=w_nz.value,
                    x_len=w_xlen.value, y_len=w_ylen.value, z_len=w_zlen.value,
                    top_depth=w_depth.value, dip=w_dip.value
                )
                layer.create_geology(
                    poro_ave=g_poro.value, perm_ave=g_perm.value,
                    poro_std=g_poro_std.value, perm_std=g_perm_std.value,
                    ntg=g_ntg.value,
                    facies_filter=(g_fx.value, g_fy.value, g_fz.value),
                    sand_filter=(g_sx.value, g_sy.value, g_sz.value),
                    nugget=g_nugget.value
                )

            elif model_type == 'Meandering Channel':
                layer = gr.MeanderingChannelLayer(
                    nx=w_nx.value, ny=w_ny.value, nz=w_nz.value,
                    x_len=w_xlen.value, y_len=w_ylen.value, z_len=w_zlen.value,
                    top_depth=w_depth.value, dip=w_dip.value
                )
                layer.create_geology(
                    channel_width=c_width.value,
                    n_channels=c_nch.value,
                    depth_width_ratio=c_dwr.value,
                    meander_scale=c_meander.value,
                    avulsion_prob=c_avulsion.value,
                    poro_ave=c_poro.value
                )

            elif model_type == 'Braided Channel':
                layer = gr.BraidedChannelLayer(
                    nx=w_nx.value, ny=w_ny.value, nz=w_nz.value,
                    x_len=w_xlen.value, y_len=w_ylen.value, z_len=w_zlen.value,
                    top_depth=w_depth.value, dip=w_dip.value
                )
                layer.create_geology(
                    braidplain_width=b_bpw.value,
                    n_channels=b_nch.value,
                    n_threads=b_nth.value,
                    depth_width_ratio=b_dwr.value,
                    bar_poro_factor=b_bar_poro.value,
                    poro_ave=b_poro.value
                )

            # Select property to plot
            prop = w_prop.value
            if prop == 'Porosity':
                data = layer.poro_mat
                label = f'{model_type} — Porosity'
            elif prop == 'Permeability':
                data = layer.perm_mat
                label = f'{model_type} — Permeability (mD)'
            else:  # Facies
                data = layer.facies.astype(float) if hasattr(layer, 'facies') else layer.active.astype(float)
                label = f'{model_type} — Facies'

            # Choose view type
            view = w_view.value
            mask_z = (prop != 'Facies')
            if view == 'Cube Slices':
                gr.plot_cube_slices(data, title=label, mask_zeros=mask_z)
            elif view == 'Z Slices':
                gr.plot_slices(data, axis=2, title=label, mask_zeros=mask_z)
            elif view == 'X Slices':
                gr.plot_slices(data, axis=0, title=label, mask_zeros=mask_z)
            else:
                gr.plot_slices(data, axis=1, title=label, mask_zeros=mask_z)
            plt.show()

            # Print summary stats
            active_mask = layer.active == 1
            print(f'Grid: ({w_nx.value}, {w_ny.value}, {w_nz.value})')
            print(f'Active fraction: {layer.active.mean():.2f}')
            if active_mask.any():
                print(f'Porosity  — mean: {layer.poro_mat[active_mask].mean():.3f}, std: {layer.poro_mat[active_mask].std():.3f}')
                print(f'Perm (mD) — mean: {layer.perm_mat[active_mask].mean():.2f}, std: {layer.perm_mat[active_mask].std():.2f}')

        except Exception as e:
            print(f'Error: {e}')
            import traceback
            traceback.print_exc()

# ── Connect all widgets to auto-generate ─────────────────────────────
all_widgets = [
    model_selector, w_nx, w_ny, w_nz, w_xlen, w_ylen, w_zlen, w_depth, w_dip,
    w_view, w_prop,
    # Lobe
    l_poro, l_perm, l_poro_std, l_perm_std, l_ntg, l_rmin, l_rmax, l_asp, l_m, l_upthin, l_bouma,
    # Gaussian
    g_poro, g_perm, g_poro_std, g_perm_std, g_ntg, g_fx, g_fy, g_fz, g_sx, g_sy, g_sz, g_nugget,
    # Meandering Channel
    c_width, c_nch, c_dwr, c_meander, c_avulsion, c_poro,
    # Braided Channel
    b_bpw, b_nch, b_nth, b_dwr, b_bar_poro, b_poro,
]
for w in all_widgets:
    w.observe(generate_model, names='value')

# ── Layout ────────────────────────────────────────────────────────────
update_params()  # show initial panel

display(widgets.VBox([
    widgets.HTML('<h2>GeoRules Interactive Explorer</h2>'),
    model_selector,
    grid_box,
    param_area,
    view_box,
    output_area
]))